<a href="https://colab.research.google.com/github/liuxiaohu0511/lance-demo/blob/develop/lance_json.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install pylance

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 MB 18.7 MB/s eta 0:00:00


In [3]:
import shutil
import lance
import numpy as np
import pandas as pd
import pyarrow as pa
import duckdb

In [5]:
!pip install --upgrade pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 MB 17.2 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pylibcudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 21.0.0 which is incompatible.
cudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 21.0.0 which is incompatible.


lance提供了json数据的存储和查询，并能高效的处理半结构数据

In [1]:
import lance
import pyarrow as pa
import json

# Create a table with JSON data
json_data = {"name": "Alice", "age": 30, "city": "New York"}
json_arr = pa.array([json.dumps(json_data)], type=pa.json_())
table = pa.table({"id": [1], "data": json_arr})

# Write the dataset
lance.write_dataset(table, "dataset.lance",mode='overwrite')

In [4]:
ds = lance.dataset("dataset.lance")
ds.to_table().to_pandas()
ds.schema

id: int64
data: large_binary
  -- field metadata --
  ARROW:extension:metadata: ''
  ARROW:extension:name: 'lance.json'

**存储格式**

lance使用lance.json扩展类型将json数据内部存储为jsonb（二进制的json）。


*   通过二进制实现高效存储
*   快速查询性能，用于嵌套字段访问
*   与apache arrow的json类型的兼容性


当从lance读取json数据时，将自动转换为arrow的json类型，方便跟数据处理pipeline无缝集成。




**json函数**

lance提供了一套全面的json函数，用于查询和过滤json数据。这些函数可以在过滤表达式中使用，例如 to_table() 、 scanner() 以及通过 DataFusion 集成进行 SQL 查询。

**json_extract**

使用jsonpath从json中取值

In [8]:
# Sample data: {"user": {"name": "Alice", "age": 30}}
result = ds.to_table(
    filter="json_extract(data, '$.name') = '\"Alice\"'"
)
result.to_pandas()
# Returns: "\"Alice\"" for strings, "30" for numbers, "true" for booleans

,id,data
0,1,"{""age"":30,""city"":""New York"",""name"":""Alice""}"


**json_get**

从 JSON 中检索字段或数组元素，将其作为 JSONB 返回以进行进一步处理。

json_get(json_column, key_or_index). key_or_index，可以是json的key也可以是json array的数组下标索引值


In [10]:
dataset = ds
# Access nested JSON by chaining json_get calls
# Sample data: {"user": {"profile": {"name": "Alice"}}}
result = dataset.to_table(
    filter="json_get_string(json_get(json_get(data, 'user'), 'profile'), 'name') = 'Alice'"
)
result

# Access array elements by index
# Sample data: ["first", "second", "third"]
result = dataset.to_table(
    filter="json_get_string(data, '0') = 'first'"  # Gets first array element
)
result

pyarrow.Table
id: int64
data: extension<arrow.json>
----
id: []
data: []

取值如何保证**类型安全**



*   json_get_string(json_column, key_or_index)：字段可以是key或者数组下标索引，返回值是string或者null（如果转换失败）
*   json_get_int(json_column, key_or_index)：提取一个整数值，进行严格的类型转换。返回64位整数，转换失败返回NULL；类型转换：使用 JSONB 的严格 to_i64() 转换：- 数字被截断为整数 - 字符串必须能解析为数字 - 布尔值：true → 1，false → 0


*   json_get_float(json_column, key_or_index)：提取一个浮点值，类型转换失败，返回NULL。返回64位浮点数。使用 JSONB 的严格 to_f64() 转换：- 整数转换为浮点数 - 字符串必须能解析为数字 - 布尔值：true → 1.0，false → 0.0
*   json_get_bool(json_column, key_or_index):提取具有严格类型转换的布尔值。如果类型转换失败，返回NULL。使用 JSONB 的严格 to_bool() 转换：- 数字：0 → false，非零 → true - 字符串："true" → true，"false" → false（需要精确匹配）- 其他值可能无法转换



*   json_exists():检查json数据中是否存在jsonPath。json_exists(json_column, json_path)
*   json_array_contains(): 检查json数组是否包含特定值。json_array_contains(json_column, json_path, value)


*   json_array_length(): 返回json数组的长度。json_array_length(json_column, json_path)。如果路径不存在，返回null；如果json_path指向非数组值，报错。















In [ ]:
result = dataset.to_table(
    filter="json_get_string(data, 'name') = 'Alice'"
)

# Array access example
# Sample data: ["first", "second"]
result = dataset.to_table(
    filter="json_get_string(data, '1') = 'second'"  # Gets second array element
)

# {"age": 30} works, {"age": "30"} may work if JSONB allows string parsing
result = dataset.to_table(
    filter="json_get_int(data, 'age') > 25"
)

result = dataset.to_table(
    filter="json_get_float(data, 'score') >= 90.5"
)

result = dataset.to_table(
    filter="json_get_bool(data, 'active') = true"
)

# Find records that have an age field
result = dataset.to_table(
    filter="json_exists(data, '$.user.age')"
)

# Sample data: {"tags": ["python", "ml", "data"]}
result = dataset.to_table(
    filter="json_array_contains(data, '$.tags', 'python')"
)

# Find records with more than 3 tags
result = dataset.to_table(
    filter="json_array_length(data, '$.tags') > 3"
)

# Empty arrays return 0
result = dataset.to_table(
    filter="json_array_length(data, '$.empty_array') = 0"
)


**处理嵌套的json**

In [24]:
import lance
import pyarrow as pa
import json

# Create nested JSON data
data = [
    {
        "id": 1,
        "user": {
            "profile": {
                "name": "Alice",
                "settings": {
                    "theme": "dark",
                    "notifications": True
                }
            },
            "scores": [95, 87, 92]
        }
    },
    {
        "id": 2,
        "user": {
            "profile": {
                "name": "Bob",
                "settings": {
                    "theme": "light",
                    "notifications": False
                }
            },
            "scores": [88, 91, 85]
        }
    }
]

# Convert to Lance dataset
json_strings = [json.dumps(d) for d in data]
table = pa.table({
    "data": pa.array(json_strings, type=pa.json_())
})

lance.write_dataset(table, "nested.lance", mode='overwrite')

In [31]:
dataset = lance.dataset("nested.lance")

dataset.versions()

dataset.to_table().to_pandas()

,data
0,"{""id"":1,""user"":{""profile"":{""name"":""Alice"",""set..."
1,"{""id"":2,""user"":{""profile"":{""name"":""Bob"",""setti..."


from matplotlib import pyplot as plt
import seaborn as sns
_df_0.groupby('data').size().plot(kind='barh', color=sns.palettes.mpl_palette('Dark2'))
plt.gca().spines[['top', 'right',]].set_visible(False)

In [39]:
# Query nested fields using JSONPath
dark_theme_users = dataset.to_table(
    filter="json_extract(data, '$.user.profile.settings.theme') = '\"dark\"'"
)
dark_theme_users.to_pandas()

,data
0,"{""id"":1,""user"":{""profile"":{""name"":""Alice"",""set..."


In [40]:
# Or using chained json_get
high_scorers = dataset.to_table(
    filter="json_array_length(data, '$.user.scores') >= 3"
)

high_scorers.to_pandas()

,data
0,"{""id"":1,""user"":{""profile"":{""name"":""Alice"",""set..."
1,"{""id"":2,""user"":{""profile"":{""name"":""Bob"",""setti..."


**结合json和其他数据类型**


In [41]:
# Create mixed-type table with JSON metadata
products = pa.table({
    "id": [1, 2, 3],
    "name": ["Laptop", "Phone", "Tablet"],
    "price": [999.99, 599.99, 399.99],
    "specs": pa.array([
        json.dumps({"cpu": "i7", "ram": 16, "storage": 512}),
        json.dumps({"screen": 6.1, "battery": 4000, "5g": True}),
        json.dumps({"screen": 10.5, "battery": 7000, "stylus": True})
    ], type=pa.json_())
})

lance.write_dataset(products, "products.lance")
dataset = lance.dataset("products.lance")

# Find products with specific specs
result = dataset.to_table(
    filter="price < 600 AND json_get_bool(specs, '5g') = true"
)

In [43]:
result.to_pandas()

,id,name,price,specs
0,2,Phone,599.99,"{""5g"":true,""battery"":4000,""screen"":6.1}"


**处理json中的数组**

In [44]:
# Create data with JSON arrays
records = pa.table({
    "id": [1, 2, 3],
    "data": pa.array([
        json.dumps({"name": "Project A", "tags": ["python", "ml", "production"]}),
        json.dumps({"name": "Project B", "tags": ["rust", "systems"]}),
        json.dumps({"name": "Project C", "tags": ["python", "web", "api", "production"]})
    ], type=pa.json_())
})

lance.write_dataset(records, "projects.lance")

,id,data
0,3,"{""name"":""Project C"",""tags"":[""python"",""web"",""ap..."


In [48]:
%%time
dataset = lance.dataset("projects.lance")

# Find projects with Python
python_projects = dataset.to_table(
    filter="json_array_contains(data, '$.tags', 'python')"
)

python_projects.to_pandas()

CPU times: user 5.86 ms, sys: 1.2 ms, total: 7.06 ms
Wall time: 6.97 ms


,id,data
0,1,"{""name"":""Project A"",""tags"":[""python"",""ml"",""pro..."
1,3,"{""name"":""Project C"",""tags"":[""python"",""web"",""ap..."


In [49]:
# Find projects with more than 3 tags
complex_projects = dataset.to_table(
    filter="json_array_length(data, '$.tags') > 3"
)
complex_projects.to_pandas()

,id,data
0,3,"{""name"":""Project C"",""tags"":[""python"",""web"",""ap..."


**性能最佳实践**



*   选择正确的函数：使用 json_get_* 函数进行直接字段访问和类型转换；使用 json_extract 进行复杂的 JSONPath 查询。
*   索引频繁查询路径：考虑为频繁访问的 JSON 路径创建计算列以提高查询性能。


*   最小化深层嵌套：虽然 Lance 支持任意嵌套，但扁平结构通常性能更好。
*   理解类型转换：`json_get_*` 函数使用**严格的类型转换**，如果数据类型不匹配，就可能导致失败。因此，在设计 schema（数据结构）时要提前做好规划。


*   数组访问：当处理 JSON 数组时，您可以使用 json_get 函数通过索引使用数字字符串（例如，“0”，“1”）来访问元素。









**与DataFusion集成**

当使用 **Lance** 搭配 **Apache DataFusion** 进行 SQL 查询时，所有的 **JSON 函数** 都可以使用。
关于如何在 SQL 环境中使用这些 JSON 函数的详细说明，请参阅 **DataFusion 集成指南（DataFusion Integration Guide）**。


**限制**


*   JSONPath支持标准的JSONPath语法，但是可能不支持所有的高级功能
*   大的JSON文档可能会影响查询性能
*   JSON函数目前只用于过滤操作，不可用于查询结果中的投影。




